# 📄 Document Forgery Detection System
### Detects: Forgery, Missing Info, Stamps, Signatures, Logos & Key Fields
**Compatible with your SIFCO/Ganador shipping document dataset**

---
**What this notebook does:**
1. Installs all dependencies
2. Mounts Google Drive & loads your PDFs
3. Extracts images from PDFs
4. Detects: stamps, signatures, logos, text fields, missing info
5. Trains a multi-label classification model
6. Runs inference on new documents
7. Generates a forgery report

> **Runtime:** Set to **GPU (T4)** → Runtime > Change runtime type > T4 GPU

## 🔧 STEP 1 — Install Dependencies

In [ ]:
# Install all required packages
!pip install -q pdf2image pillow opencv-python-headless pytesseract transformers datasets
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q ultralytics scikit-learn matplotlib seaborn timm accelerate
!pip install -q reportlab fpdf2 PyMuPDF
!apt-get install -q poppler-utils tesseract-ocr tesseract-ocr-eng 2>/dev/null

import subprocess
result = subprocess.run(['tesseract', '--version'], capture_output=True, text=True)
print('✅ Tesseract:', result.stdout.split('\n')[0])
print('✅ All dependencies installed!')

## 📁 STEP 2 — Mount Google Drive & Load Your PDFs

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ─────────────────────────────────────────────────────────────────────
# ⚙️  CONFIGURE YOUR PATHS HERE
# Upload your PDFs to Google Drive and update this path:
PDF_FOLDER = '/content/drive/MyDrive/shipping_docs'   # <-- change this
OUTPUT_DIR = '/content/doc_dataset'
MODEL_SAVE_DIR = '/content/drive/MyDrive/forgery_model'
# ─────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/images', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/annotations', exist_ok=True)

# List found PDFs
if os.path.exists(PDF_FOLDER):
    pdfs = [f for f in os.listdir(PDF_FOLDER) if f.endswith('.pdf')]
    print(f'✅ Found {len(pdfs)} PDF(s) in {PDF_FOLDER}')
    for p in pdfs:
        print(f'   📄 {p}')
else:
    print(f'⚠️  Folder not found: {PDF_FOLDER}')
    print('   Create the folder in Drive and upload your PDFs, then re-run.')

## 🖼️ STEP 3 — Extract & Preprocess PDF Pages as Images

In [ ]:
from pdf2image import convert_from_path
from PIL import Image
import numpy as np
import json, glob

def pdf_to_images(pdf_path, out_dir, dpi=200):
    """Convert each PDF page to a high-res image."""
    doc_name = os.path.splitext(os.path.basename(pdf_path))[0]
    pages = convert_from_path(pdf_path, dpi=dpi)
    saved = []
    for i, page in enumerate(pages):
        # Resize to standard size for model input
        page = page.convert('RGB')
        page = page.resize((1024, 1024), Image.LANCZOS)
        img_path = f'{out_dir}/images/{doc_name}_page{i+1}.jpg'
        page.save(img_path, 'JPEG', quality=95)
        saved.append(img_path)
    return saved

all_images = []
pdf_list = glob.glob(f'{PDF_FOLDER}/*.pdf')

if not pdf_list:
    # Demo mode: create synthetic test images if no PDFs found
    print('⚠️  No PDFs found. Running in DEMO MODE with synthetic images.')
    print('   Upload real PDFs to your Drive folder for actual training.')
    for i in range(6):
        img = Image.fromarray(np.random.randint(230, 255, (1024,1024,3), dtype=np.uint8))
        img_path = f'{OUTPUT_DIR}/images/demo_doc_{i+1}_page1.jpg'
        img.save(img_path)
        all_images.append(img_path)
else:
    for pdf in pdf_list:
        imgs = pdf_to_images(pdf, OUTPUT_DIR)
        all_images.extend(imgs)
        print(f'   ✅ {os.path.basename(pdf)} → {len(imgs)} page(s) extracted')

print(f'\n📊 Total images ready for processing: {len(all_images)}')

## 🔍 STEP 4 — Detect Document Elements (Stamps, Signatures, Logos, Text Fields)

In [ ]:
import cv2
import pytesseract
from PIL import Image
import numpy as np
import re

# ── Required fields for shipping/freight documents ──────────────────
REQUIRED_FIELDS = [
    'invoice', 'date', 'total', 'consignee', 'container',
    'bill of lading', 'freight', 'destination', 'origin',
    'signature', 'stamp', 'vessel', 'weight'
]

def detect_stamp(img_np):
    """Detect circular/rectangular stamps using Hough circles + contour analysis."""
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (9, 9), 2)
    
    # Detect circular stamps
    circles = cv2.HoughCircles(
        blurred, cv2.HOUGH_GRADIENT, dp=1.2, minDist=50,
        param1=50, param2=30, minRadius=20, maxRadius=150
    )
    
    # Also look for colored regions (stamps often blue/red)
    img_hsv = cv2.cvtColor(img_np, cv2.COLOR_RGB2HSV)
    # Blue stamp range
    blue_mask = cv2.inRange(img_hsv, (100, 50, 50), (130, 255, 255))
    # Red stamp range
    red_mask = cv2.inRange(img_hsv, (0, 50, 50), (10, 255, 255))
    color_stamp = (cv2.countNonZero(blue_mask) > 500 or cv2.countNonZero(red_mask) > 500)
    
    has_circle_stamp = circles is not None and len(circles[0]) > 0
    return has_circle_stamp or color_stamp, {
        'circle_detected': has_circle_stamp,
        'color_region_detected': color_stamp,
        'circle_count': len(circles[0]) if circles is not None else 0
    }

def detect_signature(img_np):
    """Detect handwritten signature using stroke analysis."""
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    # Focus on bottom 40% of doc (signatures usually appear there)
    h = gray.shape[0]
    bottom_region = gray[int(h*0.6):, :]
    
    thresh = cv2.adaptiveThreshold(
        bottom_region, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, 11, 2
    )
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # Signatures: many small-medium contours with high aspect ratio variability
    sig_candidates = [
        c for c in contours
        if 20 < cv2.contourArea(c) < 5000
    ]
    has_signature = len(sig_candidates) > 15
    return has_signature, {'candidate_strokes': len(sig_candidates)}

def detect_logo(img_np):
    """Detect company logo (usually top region with complex graphics)."""
    h, w = img_np.shape[:2]
    top_region = img_np[:int(h*0.25), :int(w*0.5)]
    
    gray = cv2.cvtColor(top_region, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    edge_density = np.sum(edges > 0) / edges.size
    
    # Logo = moderate-high edge density in top-left quadrant
    has_logo = edge_density > 0.02
    return has_logo, {'edge_density': round(float(edge_density), 4)}

def extract_text_and_check_fields(img_np):
    """OCR the document and check for required shipping fields."""
    pil_img = Image.fromarray(img_np)
    # Enhance for better OCR
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)
    ocr_img = Image.fromarray(thresh)
    
    raw_text = pytesseract.image_to_string(ocr_img, config='--psm 6').lower()
    
    found_fields = {}
    missing_fields = []
    for field in REQUIRED_FIELDS:
        present = field.lower() in raw_text
        found_fields[field] = present
        if not present:
            missing_fields.append(field)
    
    # Extract key values
    amounts = re.findall(r'\$?\d{1,3}(?:,\d{3})*(?:\.\d{2})?', raw_text)
    dates = re.findall(r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}', raw_text)
    
    return {
        'found_fields': found_fields,
        'missing_fields': missing_fields,
        'amounts_found': amounts[:5],
        'dates_found': dates[:3],
        'text_length': len(raw_text),
        'raw_text_preview': raw_text[:200]
    }

def analyze_document(img_path):
    """Full analysis pipeline for a single document image."""
    img = Image.open(img_path).convert('RGB')
    img_np = np.array(img)
    
    stamp_found, stamp_info = detect_stamp(img_np)
    sig_found, sig_info = detect_signature(img_np)
    logo_found, logo_info = detect_logo(img_np)
    text_info = extract_text_and_check_fields(img_np)
    
    # Forgery risk scoring
    risk_score = 0
    risk_factors = []
    
    if not stamp_found:
        risk_score += 30
        risk_factors.append('MISSING_STAMP')
    if not sig_found:
        risk_score += 25
        risk_factors.append('MISSING_SIGNATURE')
    if not logo_found:
        risk_score += 20
        risk_factors.append('MISSING_LOGO')
    if len(text_info['missing_fields']) > 4:
        risk_score += 25
        risk_factors.append('MANY_MISSING_FIELDS')
    if text_info['text_length'] < 100:
        risk_score += 20
        risk_factors.append('LOW_TEXT_CONTENT')
    
    risk_level = 'LOW' if risk_score < 30 else 'MEDIUM' if risk_score < 60 else 'HIGH'
    
    return {
        'image_path': img_path,
        'doc_name': os.path.basename(img_path),
        'stamp': {'detected': stamp_found, **stamp_info},
        'signature': {'detected': sig_found, **sig_info},
        'logo': {'detected': logo_found, **logo_info},
        'text_analysis': text_info,
        'forgery_risk': {
            'score': risk_score,
            'level': risk_level,
            'factors': risk_factors
        }
    }

print('✅ Detection functions loaded.')
print('📋 Required fields being checked:', REQUIRED_FIELDS)

## 🧪 STEP 5 — Run Analysis on All Documents

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display

results = []

print('🔍 Analyzing documents...\n')
for img_path in all_images:
    try:
        result = analyze_document(img_path)
        results.append(result)
        r = result['forgery_risk']
        s = result['stamp']['detected']
        sig = result['signature']['detected']
        logo = result['logo']['detected']
        miss = result['text_analysis']['missing_fields']
        
        print(f"📄 {result['doc_name']}")
        print(f"   Stamp: {'✅' if s else '❌'}  Signature: {'✅' if sig else '❌'}  Logo: {'✅' if logo else '❌'}")
        print(f"   Missing fields: {miss if miss else 'None ✅'}")
        print(f"   Risk: [{r['level']}] score={r['score']}  factors={r['factors']}")
        print()
    except Exception as e:
        print(f'   ⚠️  Error on {os.path.basename(img_path)}: {e}')

# Save results
with open(f'{OUTPUT_DIR}/analysis_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'💾 Results saved to {OUTPUT_DIR}/analysis_results.json')

## 📊 STEP 6 — Visualize Detection Results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

def visualize_document_analysis(result):
    """Show document image with detection overlays and summary."""
    img = Image.open(result['image_path']).convert('RGB')
    img_np = np.array(img)
    
    fig = plt.figure(figsize=(16, 8))
    gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1])
    
    # Document image
    ax1 = fig.add_subplot(gs[0])
    ax1.imshow(img_np)
    ax1.set_title(result['doc_name'], fontsize=10, fontweight='bold')
    ax1.axis('off')
    
    # Draw detection zones
    h, w = img_np.shape[:2]
    # Stamp region indicator
    stamp_color = 'green' if result['stamp']['detected'] else 'red'
    rect = plt.Rectangle((w*0.6, h*0.7), w*0.35, h*0.25,
                           linewidth=2, edgecolor=stamp_color, facecolor='none',
                           linestyle='--')
    ax1.add_patch(rect)
    ax1.text(w*0.62, h*0.68, 'STAMP ZONE', color=stamp_color, fontsize=7, fontweight='bold')
    
    # Logo region
    logo_color = 'green' if result['logo']['detected'] else 'red'
    rect2 = plt.Rectangle((0, 0), w*0.45, h*0.22,
                            linewidth=2, edgecolor=logo_color, facecolor='none',
                            linestyle='--')
    ax1.add_patch(rect2)
    ax1.text(5, h*0.24, 'LOGO ZONE', color=logo_color, fontsize=7, fontweight='bold')
    
    # Signature region
    sig_color = 'green' if result['signature']['detected'] else 'red'
    rect3 = plt.Rectangle((0, h*0.75), w, h*0.22,
                            linewidth=2, edgecolor=sig_color, facecolor='none',
                            linestyle='--')
    ax1.add_patch(rect3)
    ax1.text(5, h*0.74, 'SIGNATURE ZONE', color=sig_color, fontsize=7, fontweight='bold')
    
    # Summary panel
    ax2 = fig.add_subplot(gs[1])
    ax2.axis('off')
    
    risk = result['forgery_risk']
    risk_colors = {'LOW': '#2ecc71', 'MEDIUM': '#f39c12', 'HIGH': '#e74c3c'}
    risk_color = risk_colors[risk['level']]
    
    summary_lines = [
        (f"FORGERY RISK: {risk['level']}", 18, risk_color, 'bold'),
        (f"Risk Score: {risk['score']}/100", 13, 'black', 'normal'),
        ('', 10, 'black', 'normal'),
        ('ELEMENT DETECTION', 12, '#2c3e50', 'bold'),
        (f"  {'✅' if result['stamp']['detected'] else '❌'} Stamp/Seal", 11, 'black', 'normal'),
        (f"  {'✅' if result['signature']['detected'] else '❌'} Signature", 11, 'black', 'normal'),
        (f"  {'✅' if result['logo']['detected'] else '❌'} Company Logo", 11, 'black', 'normal'),
        ('', 10, 'black', 'normal'),
        ('MISSING FIELDS', 12, '#2c3e50', 'bold'),
    ]
    
    missing = result['text_analysis']['missing_fields']
    if missing:
        for mf in missing[:6]:
            summary_lines.append((f'  ❌ {mf}', 10, '#c0392b', 'normal'))
    else:
        summary_lines.append(('  ✅ All key fields found', 10, '#27ae60', 'normal'))
    
    if risk['factors']:
        summary_lines.append(('', 10, 'black', 'normal'))
        summary_lines.append(('RISK FACTORS', 12, '#2c3e50', 'bold'))
        for f in risk['factors']:
            summary_lines.append((f'  ⚠️  {f}', 10, '#e67e22', 'normal'))
    
    y_pos = 0.98
    for text, size, color, weight in summary_lines:
        ax2.text(0.05, y_pos, text, transform=ax2.transAxes,
                 fontsize=size, color=color, fontweight=weight, va='top')
        y_pos -= 0.06
    
    # Risk bar
    bar_ax = fig.add_axes([0.55, 0.08, 0.4, 0.04])
    bar_ax.barh([0], [risk['score']], color=risk_color, height=0.5)
    bar_ax.barh([0], [100], color='#ecf0f1', height=0.5, left=0)
    bar_ax.barh([0], [risk['score']], color=risk_color, height=0.5)
    bar_ax.set_xlim(0, 100)
    bar_ax.set_yticks([])
    bar_ax.set_xlabel('Risk Score', fontsize=8)
    bar_ax.axvline(x=30, color='orange', linestyle='--', linewidth=1, alpha=0.7)
    bar_ax.axvline(x=60, color='red', linestyle='--', linewidth=1, alpha=0.7)
    
    plt.suptitle('Document Forgery Analysis Report', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/{result['doc_name'].replace('.jpg','')}_analysis.png",
                bbox_inches='tight', dpi=150)
    plt.show()

# Show analysis for each document
for result in results:
    visualize_document_analysis(result)

## 🏗️ STEP 7 — Build & Train the Deep Learning Model

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from sklearn.model_selection import train_test_split
import numpy as np

# ── Label schema (multi-label classification) ───────────────────────
LABELS = ['has_stamp', 'has_signature', 'has_logo',
          'has_missing_fields', 'high_forgery_risk']
NUM_LABELS = len(LABELS)

print(f'🏷️  Labels ({NUM_LABELS}): {LABELS}')

# ── Dataset class ────────────────────────────────────────────────────
class DocumentDataset(Dataset):
    def __init__(self, results, transform=None, augment=True):
        self.data = results
        self.augment = augment
        base_transform = [
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ]
        if augment:
            self.transform = transforms.Compose([
                transforms.Resize((256, 256)),
                transforms.RandomCrop(224),
                transforms.RandomHorizontalFlip(p=0.3),
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
            ])
        else:
            self.transform = transforms.Compose(base_transform)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        r = self.data[idx]
        img = Image.open(r['image_path']).convert('RGB')
        img = self.transform(img)

        # Build label vector from analysis results
        label = torch.tensor([
            float(r['stamp']['detected']),
            float(r['signature']['detected']),
            float(r['logo']['detected']),
            float(len(r['text_analysis']['missing_fields']) > 2),
            float(r['forgery_risk']['level'] == 'HIGH')
        ], dtype=torch.float32)
        return img, label

# ── Model: EfficientNet-B0 with custom head ──────────────────────────
class DocumentForgeryModel(nn.Module):
    def __init__(self, num_labels=NUM_LABELS):
        super().__init__()
        # Use pretrained EfficientNet as backbone
        from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
        backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_features = backbone.classifier[1].in_features
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_labels)
            # No sigmoid here — BCEWithLogitsLoss handles it
        )
    
    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DocumentForgeryModel().to(device)
print(f'✅ Model loaded on: {device}')
print(f'   Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── Training setup ───────────────────────────────────────────────────
from torch.optim.lr_scheduler import CosineAnnealingLR

# Split data (with small datasets, use leave-one-out or k-fold)
if len(results) >= 4:
    train_data, val_data = train_test_split(results, test_size=0.2, random_state=42)
else:
    # With very few samples, train on all and validate on all (for demo)
    train_data, val_data = results, results
    print('⚠️  Small dataset: training and validating on all samples.')
    print('   Add more labeled documents for better generalization.')

# Data augmentation: repeat small datasets to create more training samples
if len(train_data) < 20:
    train_data = train_data * (20 // len(train_data) + 1)
    print(f'   Augmented training set to {len(train_data)} samples via repetition.')

train_dataset = DocumentDataset(train_data, augment=True)
val_dataset   = DocumentDataset(val_data,   augment=False)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=20, eta_min=1e-6)

EPOCHS = 20
train_losses, val_losses = [], []
best_val_loss = float('inf')

print(f'🚀 Starting training for {EPOCHS} epochs...')
print(f'   Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')
print(f'   Loss: BCEWithLogitsLoss | Optimizer: AdamW | LR: 1e-4')
print()

In [ ]:
# ── Training loop ────────────────────────────────────────────────────
from sklearn.metrics import f1_score

for epoch in range(EPOCHS):
    # TRAIN
    model.train()
    train_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    # VALIDATE
    model.eval()
    val_loss = 0
    all_preds, all_targets = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            preds = torch.sigmoid(outputs) > 0.5
            all_preds.append(preds.cpu().numpy())
            all_targets.append(labels.cpu().numpy())
    
    avg_train = train_loss / len(train_loader)
    avg_val   = val_loss   / len(val_loader)
    train_losses.append(avg_train)
    val_losses.append(avg_val)
    scheduler.step()
    
    # F1 score
    all_preds   = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)
    f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    
    # Save best model
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), f'{MODEL_SAVE_DIR}/best_model.pth')
        saved_str = ' 💾 SAVED'
    else:
        saved_str = ''
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch [{epoch+1:02d}/{EPOCHS}] '
              f'Train: {avg_train:.4f} | Val: {avg_val:.4f} | '
              f'F1: {f1:.3f}{saved_str}')

print(f'\n✅ Training complete! Best val loss: {best_val_loss:.4f}')
print(f'💾 Best model saved to: {MODEL_SAVE_DIR}/best_model.pth')

## 📈 STEP 8 — Training Curves & Performance Metrics

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Loss curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, label='Train Loss', color='#3498db', linewidth=2)
axes[0].plot(val_losses, label='Val Loss', color='#e74c3c', linewidth=2)
axes[0].set_title('Training & Validation Loss', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCE Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Per-label performance
model.load_state_dict(torch.load(f'{MODEL_SAVE_DIR}/best_model.pth', map_location=device))
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        outputs = torch.sigmoid(model(imgs)) > 0.5
        all_preds.append(outputs.cpu().numpy())
        all_targets.append(labels.numpy())

all_preds   = np.vstack(all_preds)
all_targets = np.vstack(all_targets)

f1_per_label = f1_score(all_targets, all_preds, average=None, zero_division=0)
colors = ['#2ecc71' if f > 0.7 else '#f39c12' if f > 0.5 else '#e74c3c' for f in f1_per_label]
axes[1].barh(LABELS, f1_per_label, color=colors)
axes[1].set_title('F1 Score per Label', fontweight='bold')
axes[1].set_xlabel('F1 Score')
axes[1].set_xlim(0, 1)
for i, v in enumerate(f1_per_label):
    axes[1].text(v + 0.02, i, f'{v:.2f}', va='center', fontsize=10)
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 Classification Report:')
print(classification_report(all_targets, all_preds, target_names=LABELS, zero_division=0))

## 🚀 STEP 9 — Inference on New Documents

In [ ]:
def predict_document(pdf_path_or_img_path, model, device):
    """
    Run full forgery analysis on a new document.
    Accepts either a PDF path or an image path.
    """
    # Convert PDF if needed
    if pdf_path_or_img_path.endswith('.pdf'):
        pages = convert_from_path(pdf_path_or_img_path, dpi=200)
        img = pages[0].convert('RGB').resize((1024, 1024))
        tmp_path = '/tmp/tmp_doc_page.jpg'
        img.save(tmp_path)
        img_path = tmp_path
    else:
        img_path = pdf_path_or_img_path

    # Rule-based analysis
    rule_result = analyze_document(img_path)

    # Deep learning prediction
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    img = Image.open(img_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(img_tensor)
        probs  = torch.sigmoid(logits).cpu().numpy()[0]
        preds  = probs > 0.5

    dl_result = {label: {'prob': float(p), 'detected': bool(d)}
                 for label, p, d in zip(LABELS, probs, preds)}

    # Combine rule-based + DL forgery risk
    combined_risk = rule_result['forgery_risk'].copy()
    if dl_result['high_forgery_risk']['prob'] > 0.6:
        combined_risk['dl_flag'] = 'HIGH_RISK_DETECTED_BY_MODEL'
    
    print(f"\n{'='*55}")
    print(f"DOCUMENT: {os.path.basename(pdf_path_or_img_path)}")
    print(f"{'='*55}")
    print(f"\n📌 RULE-BASED DETECTION")
    print(f"   Stamp:      {'✅ FOUND' if rule_result['stamp']['detected'] else '❌ NOT FOUND'}")
    print(f"   Signature:  {'✅ FOUND' if rule_result['signature']['detected'] else '❌ NOT FOUND'}")
    print(f"   Logo:       {'✅ FOUND' if rule_result['logo']['detected'] else '❌ NOT FOUND'}")
    print(f"   Missing:    {rule_result['text_analysis']['missing_fields'] or 'None'}")

    print(f"\n🤖 DEEP LEARNING PREDICTIONS")
    for label, info in dl_result.items():
        bar = '█' * int(info['prob'] * 20)
        print(f"   {label:<22} {bar:<20} {info['prob']:.1%}")

    risk = combined_risk
    print(f"\n⚠️  FORGERY RISK: [{risk['level']}] Score: {risk['score']}/100")
    if risk['factors']:
        for f in risk['factors']:
            print(f"   → {f}")
    print()
    
    return {'rule_based': rule_result, 'deep_learning': dl_result, 'risk': combined_risk}

# ── Run on all extracted images ──────────────────────────────────────
print('🔎 Running inference on all documents...\n')
for img_path in all_images:
    predict_document(img_path, model, device)

## 📝 STEP 10 — Generate PDF Forgery Report

In [ ]:
from fpdf import FPDF
from datetime import datetime

class ForgeryReport(FPDF):
    def header(self):
        self.set_font('Helvetica', 'B', 14)
        self.set_fill_color(30, 50, 80)
        self.set_text_color(255, 255, 255)
        self.cell(0, 12, 'DOCUMENT FORGERY DETECTION REPORT', fill=True, ln=True, align='C')
        self.set_text_color(0, 0, 0)
        self.set_font('Helvetica', '', 9)
        self.cell(0, 6, f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")} | System: SIFCO Doc Verifier v1.0',
                  ln=True, align='C')
        self.ln(4)

    def footer(self):
        self.set_y(-15)
        self.set_font('Helvetica', 'I', 8)
        self.cell(0, 10, f'Page {self.page_no()}', align='C')

def generate_report(results_list, output_path):
    pdf = ForgeryReport()
    pdf.add_page()
    pdf.set_auto_page_break(auto=True, margin=15)

    # Summary table header
    pdf.set_font('Helvetica', 'B', 11)
    pdf.set_fill_color(240, 240, 240)
    pdf.cell(0, 8, 'SUMMARY OF ALL ANALYZED DOCUMENTS', fill=True, ln=True)
    pdf.ln(2)

    for r in results_list:
        risk = r['forgery_risk']
        risk_colors = {'LOW': (46, 204, 113), 'MEDIUM': (243, 156, 18), 'HIGH': (231, 76, 60)}
        rc = risk_colors.get(risk['level'], (100, 100, 100))

        pdf.set_font('Helvetica', 'B', 10)
        pdf.cell(0, 7, f"Document: {r['doc_name']}", ln=True)

        pdf.set_font('Helvetica', '', 9)
        col_w = 45
        items = [
            ('Stamp', '✓ FOUND' if r['stamp']['detected'] else '✗ MISSING'),
            ('Signature', '✓ FOUND' if r['signature']['detected'] else '✗ MISSING'),
            ('Logo', '✓ FOUND' if r['logo']['detected'] else '✗ MISSING'),
            ('Missing Fields', str(len(r['text_analysis']['missing_fields']))),
        ]
        for label, val in items:
            pdf.set_fill_color(248, 248, 248)
            pdf.cell(col_w, 6, label + ':', fill=True)
            pdf.cell(col_w, 6, val, ln=False)
        pdf.ln(6)

        # Risk badge
        pdf.set_fill_color(*rc)
        pdf.set_text_color(255, 255, 255)
        pdf.set_font('Helvetica', 'B', 9)
        pdf.cell(60, 7, f"RISK: {risk['level']} ({risk['score']}/100)", fill=True, ln=False)
        pdf.set_text_color(0, 0, 0)
        if risk['factors']:
            pdf.set_font('Helvetica', '', 8)
            pdf.cell(0, 7, '  Factors: ' + ', '.join(risk['factors']), ln=True)
        else:
            pdf.ln(7)

        if r['text_analysis']['missing_fields']:
            pdf.set_font('Helvetica', 'I', 8)
            pdf.set_text_color(180, 0, 0)
            pdf.cell(0, 5, 'Missing: ' + ', '.join(r['text_analysis']['missing_fields']), ln=True)
            pdf.set_text_color(0, 0, 0)

        pdf.set_draw_color(200, 200, 200)
        pdf.line(pdf.get_x(), pdf.get_y() + 1, 200, pdf.get_y() + 1)
        pdf.ln(5)

    pdf.output(output_path)
    print(f'✅ Report saved: {output_path}')

report_path = f'{OUTPUT_DIR}/forgery_report.pdf'
generate_report(results, report_path)

# Copy to Drive
import shutil
shutil.copy(report_path, f'{MODEL_SAVE_DIR}/forgery_report.pdf')
print(f'✅ Report also saved to Google Drive: {MODEL_SAVE_DIR}/forgery_report.pdf')

## 💾 STEP 11 — Save Full Model & Export for Deployment

In [ ]:
import torch
import json

# Load best weights
model.load_state_dict(torch.load(f'{MODEL_SAVE_DIR}/best_model.pth', map_location=device))
model.eval()

# Save complete model package
torch.save({
    'model_state_dict': model.state_dict(),
    'labels': LABELS,
    'num_labels': NUM_LABELS,
    'required_fields': REQUIRED_FIELDS,
    'model_architecture': 'EfficientNet-B0',
    'input_size': (224, 224),
    'normalize_mean': [0.485, 0.456, 0.406],
    'normalize_std': [0.229, 0.224, 0.225],
}, f'{MODEL_SAVE_DIR}/forgery_model_complete.pth')

# Export ONNX for production deployment
dummy_input = torch.randn(1, 3, 224, 224).to(device)
torch.onnx.export(
    model, dummy_input,
    f'{MODEL_SAVE_DIR}/forgery_model.onnx',
    input_names=['document_image'],
    output_names=['label_logits'],
    dynamic_axes={'document_image': {0: 'batch'}},
    opset_version=11
)

# Save metadata
metadata = {
    'labels': LABELS,
    'required_fields': REQUIRED_FIELDS,
    'input_size': [224, 224],
    'threshold': 0.5,
    'architecture': 'EfficientNet-B0 + Multi-label head',
    'training_samples': len(train_data),
    'best_val_loss': best_val_loss
}
with open(f'{MODEL_SAVE_DIR}/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('✅ All model files saved to Google Drive:')
print(f'   📦 forgery_model_complete.pth  — PyTorch full model')
print(f'   📦 best_model.pth              — Best weights only')
print(f'   📦 forgery_model.onnx          — ONNX for production')
print(f'   📋 model_metadata.json         — Config & labels')
print(f'   📄 forgery_report.pdf          — Document analysis report')

## 🗂️ STEP 12 — How to Add More Training Data

To improve model accuracy with more labeled examples:

```python
# Add this cell to label new documents manually
MANUAL_LABELS = {
    'UNIQUE_HYBRID_page1.jpg':        [1, 0, 1, 0, 0],  # [stamp, sig, logo, missing, high_risk]
    'shippimg_agreement_J0HN_page1.jpg': [1, 1, 1, 0, 0],
    'JOHN_SEA_FREIGHT_page1.jpg':     [1, 0, 1, 0, 0],
    # Add your own here...
}
```

**Tips for growing your dataset:**
- Collect authentic documents → label `high_forgery_risk = 0`
- Create modified/tampered copies → label `high_forgery_risk = 1`
- Aim for 50+ documents per class for robust training
- Use Roboflow or Label Studio for annotation workflows